# Bike Sharing Demand Prediction -- Washington DC

---

## Problem Statement

Bike-sharing systems provide an affordable, convenient, and eco-friendly mode of urban transportation. In Washington DC, the Capital Bikeshare system allows users to rent a bicycle from one station and return it to another. Accurately predicting the hourly demand for bike rentals is critical for operations teams who need to rebalance bikes across stations, plan fleet maintenance, and ensure availability during peak hours.

The dataset spans two years (2011-2012) of hourly rental data along with weather and seasonal information. The goal is to predict the total count of bikes rented during each hour in the test set, using only information available prior to the rental period.

**Business Value**

- Station rebalancing: Knowing when demand will spike helps logistics teams pre-position bikes where they are needed most.
- Revenue optimization: Better availability during high-demand periods directly translates to more completed rentals.
- Resource planning: Maintenance crews can schedule bike servicing during predicted low-demand windows.

**Dataset Description**

Training data (train_bikes.csv) contains 10886 hourly records with the following attributes:

- datetime: Hourly date and timestamp
- season: 1 = Spring, 2 = Summer, 3 = Fall, 4 = Winter
- holiday: Whether the day is a public holiday
- workingday: Whether the day is a working day (not weekend, not holiday)
- weather: 1 = Clear, 2 = Mist/Cloudy, 3 = Light Rain/Snow, 4 = Heavy Rain/Snow
- temp: Temperature in Celsius
- atemp: "Feels like" temperature in Celsius
- humidity: Relative humidity percentage
- windspeed: Wind speed
- casual: Count of non-registered (casual) user rentals
- registered: Count of registered user rentals
- count: Total rental count (target variable = casual + registered)

Test data (test_bikes.csv) contains 6493 records with the same features except casual, registered, and count.

**Notebook Coverage**

- Temporal feature engineering from datetime
- Thorough exploratory analysis of demand patterns
- Log transformation of the skewed target variable
- Sklearn Pipelines for reproducible preprocessing
- Multiple regression models including ensemble and boosting approaches
- Stacking and voting ensemble techniques
- Hyperparameter tuning via GridSearchCV
- Evaluation using RMSLE (competition metric), RMSE, and MAE

## Environment Setup

In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     KFold, GridSearchCV)
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             mean_squared_log_error, r2_score)
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              AdaBoostRegressor, BaggingRegressor,
                              ExtraTreesRegressor, VotingRegressor,
                              StackingRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

plt.rcParams['figure.dpi'] = 100
sns.set_palette('deep')
print("Libraries loaded successfully.")

Libraries loaded successfully.


## Data Ingestion

In [5]:
train = pd.read_csv('train_bikes.csv', parse_dates=['datetime'])
test = pd.read_csv('test_bikes.csv', parse_dates=['datetime'])

print("Training data: {} rows, {} columns".format(train.shape[0], train.shape[1]))
print("Test data:     {} rows, {} columns".format(test.shape[0], test.shape[1]))
print()
train.head(10)

Training data: 10886 rows, 12 columns
Test data:     6493 rows, 9 columns



,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0000,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0000,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0000,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0000,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0000,0,1,1
5,2011-01-01 05:00:00,1,0,0,2,9.84,12.880,75,6.0032,0,1,1
6,2011-01-01 06:00:00,1,0,0,1,9.02,13.635,80,0.0000,2,0,2
7,2011-01-01 07:00:00,1,0,0,1,8.20,12.880,86,0.0000,1,2,3
8,2011-01-01 08:00:00,1,0,0,1,9.84,14.395,75,0.0000,1,7,8
9,2011-01-01 09:00:00,1,0,0,1,13.12,17.425,76,0.0000,8,6,14


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   datetime    10886 non-null  datetime64[ns]
 1   season      10886 non-null  int64         
 2   holiday     10886 non-null  int64         
 3   workingday  10886 non-null  int64         
 4   weather     10886 non-null  int64         
 5   temp        10886 non-null  float64       
 6   atemp       10886 non-null  float64       
 7   humidity    10886 non-null  int64         
 8   windspeed   10886 non-null  float64       
 9   casual      10886 non-null  int64         
 10  registered  10886 non-null  int64         
 11  count       10886 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(8)
memory usage: 1020.7 KB


In [7]:
train.describe()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
count,10886,10886.000000,10886.000000,10886.000000,10886.000000,10886.00000,10886.000000,10886.000000,10886.000000,10886.000000,10886.000000,10886.000000
mean,2011-12-27 05:56:22.399411968,2.506614,0.028569,0.680875,1.418427,20.23086,23.655084,61.886460,12.799395,36.021955,155.552177,191.574132
min,2011-01-01 00:00:00,1.000000,0.000000,0.000000,1.000000,0.82000,0.760000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,2011-07-02 07:15:00,2.000000,0.000000,0.000000,1.000000,13.94000,16.665000,47.000000,7.001500,4.000000,36.000000,42.000000
50%,2012-01-01 20:30:00,3.000000,0.000000,1.000000,1.000000,20.50000,24.240000,62.000000,12.998000,17.000000,118.000000,145.000000
75%,2012-07-01 12:45:00,4.000000,0.000000,1.000000,2.000000,26.24000,31.060000,77.000000,16.997900,49.000000,222.000000,284.000000
max,2012-12-19 23:00:00,4.000000,1.000000,1.000000,4.000000,41.00000,45.455000,100.000000,56.996900,367.000000,886.000000,977.000000
std,NaN,1.116174,0.166599,0.466159,0.633839,7.79159,8.474601,19.245033,8.164537,49.960477,151.039033,181.144454


In [8]:
# Data quality checks
print("Missing values in training set:", train.isnull().sum().sum())
print("Missing values in test set:    ", test.isnull().sum().sum())
print("Duplicate rows in training set:", train.duplicated().sum())

Missing values in training set: 0
Missing values in test set:     0
Duplicate rows in training set: 0


## Feature Engineering from DateTime

The raw datetime column encodes rich temporal information. We extract hour, day of week, month, and year to capture the cyclical demand patterns inherent in bike sharing usage.

In [10]:
def extract_temporal_features(df):
    df = df.copy()
    df['hour'] = df['datetime'].dt.hour
    df['day_of_week'] = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month
    df['year'] = df['datetime'].dt.year
    df['day'] = df['datetime'].dt.day
    # Time-of-day bins: night(0-5), morning(6-12), afternoon(13-18), evening(19-23)
    df['time_bin'] = pd.cut(df['hour'], bins=[-1, 5, 12, 18, 23],
                            labels=[0, 1, 2, 3]).astype(int)
    # Rush hour flag
    df['is_rush_hour'] = ((df['hour'].isin([7, 8, 9, 17, 18, 19])) &
                          (df['workingday'] == 1)).astype(int)
    return df

train = extract_temporal_features(train)
test = extract_temporal_features(test)

print("New features added: hour, day_of_week, month, year, day, time_bin, is_rush_hour")
print("Training columns:", train.shape[1])
train[['datetime', 'hour', 'day_of_week', 'month', 'year', 'time_bin', 'is_rush_hour']].head(10)

New features added: hour, day_of_week, month, year, day, time_bin, is_rush_hour
Training columns: 19


,datetime,hour,day_of_week,month,year,time_bin,is_rush_hour
0,2011-01-01 00:00:00,0,5,1,2011,0,0
1,2011-01-01 01:00:00,1,5,1,2011,0,0
2,2011-01-01 02:00:00,2,5,1,2011,0,0
3,2011-01-01 03:00:00,3,5,1,2011,0,0
4,2011-01-01 04:00:00,4,5,1,2011,0,0
5,2011-01-01 05:00:00,5,5,1,2011,0,0
6,2011-01-01 06:00:00,6,5,1,2011,1,0
7,2011-01-01 07:00:00,7,5,1,2011,1,0
8,2011-01-01 08:00:00,8,5,1,2011,1,0
9,2011-01-01 09:00:00,9,5,1,2011,1,0


## Exploratory Data Analysis

### Target Variable Distribution

In [13]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Raw count distribution
axes[0].hist(train['count'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Bike Rental Count', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(train['count'].mean(), color='red', linestyle='--',
                label='Mean: {:.0f}'.format(train['count'].mean()))
axes[0].legend()

# Log-transformed count
axes[1].hist(np.log1p(train['count']), bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribution of log(1 + Count)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('log(1 + Count)')
axes[1].set_ylabel('Frequency')

# Casual vs registered
axes[2].hist(train['casual'], bins=40, alpha=0.5, color='orange', edgecolor='black', label='Casual')
axes[2].hist(train['registered'], bins=40, alpha=0.5, color='steelblue', edgecolor='black', label='Registered')
axes[2].set_title('Casual vs Registered Users', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Count')
axes[2].legend()

plt.tight_layout()
plt.show()

print("The rental count is right-skewed. Log transformation produces a more")
print("normally distributed target, which benefits many regression algorithms.")

The rental count is right-skewed. Log transformation produces a more
normally distributed target, which benefits many regression algorithms.


### Hourly Demand Patterns

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Hourly demand: working day vs non-working day
hourly_wd = train.groupby(['hour', 'workingday'])['count'].mean().unstack()
hourly_wd.columns = ['Non-Working Day', 'Working Day']
hourly_wd.plot(ax=axes[0], linewidth=2, marker='o', markersize=4)
axes[0].set_title('Average Hourly Demand: Working vs Non-Working Days',
                 fontsize=12, fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Average Rentals')
axes[0].set_xticks(range(0, 24))

# Hourly boxplots
hour_data = [train[train['hour'] == h]['count'].values for h in range(24)]
bp = axes[1].boxplot(hour_data, labels=range(24), patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.5)
axes[1].set_title('Rental Count Distribution by Hour', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print("Working days show clear bimodal peaks at morning (8 AM) and evening (5-6 PM)")
print("commute hours. Non-working days peak around midday (12-3 PM).")

Working days show clear bimodal peaks at morning (8 AM) and evening (5-6 PM)
commute hours. Non-working days peak around midday (12-3 PM).


### Seasonal and Monthly Patterns

In [17]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Monthly average
monthly = train.groupby('month')['count'].mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[0].bar(month_labels, monthly.values, color='steelblue', edgecolor='black')
axes[0].set_title('Average Demand by Month', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Average Rentals')
axes[0].tick_params(axis='x', rotation=45)

# Seasonal average
season_labels = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
seasonal = train.groupby('season')['count'].mean()
axes[1].bar([season_labels[s] for s in seasonal.index], seasonal.values,
            color=['#2ecc71', '#e74c3c', '#f39c12', '#3498db'], edgecolor='black')
axes[1].set_title('Average Demand by Season', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Average Rentals')

# Year comparison
yearly_monthly = train.groupby(['month', 'year'])['count'].mean().unstack()
yearly_monthly.plot(kind='line', ax=axes[2], marker='o', linewidth=2)
axes[2].set_title('Monthly Demand Trend: 2011 vs 2012', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Average Rentals')
axes[2].set_xticks(range(1, 13))
axes[2].legend(title='Year')

plt.tight_layout()
plt.show()

### Weather Impact on Demand

In [19]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Temperature vs count
axes[0][0].scatter(train['temp'], train['count'], alpha=0.15, s=8, color='steelblue')
axes[0][0].set_title('Temperature vs Rental Count', fontsize=12, fontweight='bold')
axes[0][0].set_xlabel('Temperature (C)')
axes[0][0].set_ylabel('Count')

# Humidity vs count
axes[0][1].scatter(train['humidity'], train['count'], alpha=0.15, s=8, color='coral')
axes[0][1].set_title('Humidity vs Rental Count', fontsize=12, fontweight='bold')
axes[0][1].set_xlabel('Humidity (%)')
axes[0][1].set_ylabel('Count')

# Windspeed vs count
axes[1][0].scatter(train['windspeed'], train['count'], alpha=0.15, s=8, color='green')
axes[1][0].set_title('Windspeed vs Rental Count', fontsize=12, fontweight='bold')
axes[1][0].set_xlabel('Windspeed')
axes[1][0].set_ylabel('Count')

# Weather category boxplot
weather_labels = {1: 'Clear', 2: 'Mist', 3: 'Light Rain', 4: 'Heavy Rain'}
weather_data = []
weather_names = []
for w in sorted(train['weather'].unique()):
    weather_data.append(train[train['weather'] == w]['count'].values)
    weather_names.append(weather_labels.get(w, str(w)))
bp = axes[1][1].boxplot(weather_data, labels=weather_names, patch_artist=True)
colors_bp = ['#2ecc71', '#f39c12', '#e74c3c', '#8e44ad']
for patch, color in zip(bp['boxes'], colors_bp[:len(bp['boxes'])]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1][1].set_title('Rental Count by Weather Category', fontsize=12, fontweight='bold')
axes[1][1].set_ylabel('Count')

plt.suptitle('Weather Factors and Bike Demand', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Correlation Analysis

In [21]:
corr_cols = ['temp', 'atemp', 'humidity', 'windspeed', 'season', 'weather',
            'holiday', 'workingday', 'hour', 'month', 'year', 'count']
corr_matrix = train[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key correlations with count:")
print(corr_matrix['count'].drop('count').sort_values(ascending=False).round(3).to_string())

Key correlations with count:
hour          0.401
temp          0.394
atemp         0.390
year          0.260
month         0.167
season        0.163
windspeed     0.101
workingday    0.012
holiday      -0.005
weather      -0.129
humidity     -0.317


### Average Demand by Temperature Bins

In [23]:
# Mean count by temperature grouping
temp_groups = train.groupby(train['temp'].round(0))[['count']].mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(temp_groups.index, temp_groups['count'], color='steelblue', linewidth=2, marker='o', markersize=3)
ax.fill_between(temp_groups.index, temp_groups['count'], alpha=0.15, color='steelblue')
ax.set_title('Average Bike Demand by Temperature', fontsize=13, fontweight='bold')
ax.set_xlabel('Temperature (C)')
ax.set_ylabel('Average Count')
plt.tight_layout()
plt.show()

## Data Preparation for Modeling

We prepare the data by selecting appropriate features, applying log transformation to the target variable, and building a preprocessing pipeline. The casual and registered columns are dropped since they are not available at prediction time.

In [25]:
# Define feature set (drop leakage columns and datetime)
drop_cols = ['datetime', 'casual', 'registered', 'count']
feature_cols = [c for c in train.columns if c not in drop_cols]

X = train[feature_cols].copy()
y = train['count'].copy()

# Log-transform the target to handle skewness
y_log = np.log1p(y)

print("Features ({} total): {}".format(len(feature_cols), feature_cols))
print()
print("Target statistics (original):")
print("  Mean: {:.2f}, Std: {:.2f}, Min: {}, Max: {}".format(
    y.mean(), y.std(), y.min(), y.max()))
print()
print("Target statistics (log-transformed):")
print("  Mean: {:.2f}, Std: {:.2f}, Min: {:.2f}, Max: {:.2f}".format(
    y_log.mean(), y_log.std(), y_log.min(), y_log.max()))

Features (15 total): ['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'hour', 'day_of_week', 'month', 'year', 'day', 'time_bin', 'is_rush_hour']

Target statistics (original):
  Mean: 191.57, Std: 181.14, Min: 1, Max: 977

Target statistics (log-transformed):
  Mean: 4.59, Std: 1.42, Min: 0.69, Max: 6.89


In [26]:
# Identify feature types
continuous_feats = ['temp', 'atemp', 'humidity', 'windspeed']
categorical_feats = ['season', 'holiday', 'workingday', 'weather', 'hour',
                     'day_of_week', 'month', 'year', 'day', 'time_bin', 'is_rush_hour']

# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), continuous_feats),
        ('cat', 'passthrough', categorical_feats)
    ]
)

print("Pipeline configured:")
print("  Continuous features ({}) -> StandardScaler".format(len(continuous_feats)))
print("  Categorical features ({}) -> Passthrough (already encoded)".format(len(categorical_feats)))

Pipeline configured:
  Continuous features (4) -> StandardScaler
  Categorical features (11) -> Passthrough (already encoded)


In [27]:
# Train-validation split
X_train, X_val, y_train, y_val, y_train_log, y_val_log = train_test_split(
    X, y, y_log, test_size=0.2, random_state=42)

# Preprocess
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)

print("Training set:   {} samples".format(X_train_proc.shape[0]))
print("Validation set: {} samples".format(X_val_proc.shape[0]))

Training set:   8708 samples
Validation set: 2178 samples


## RMSLE Evaluation Metric

The standard competition metric for bike sharing demand is Root Mean Squared Logarithmic Error (RMSLE). Since we model log(1+count) directly, RMSLE on the original scale equals RMSE on the log-transformed scale.

In [29]:
def rmsle(actual, predicted):
    # Clip predictions to avoid log of negative numbers
    predicted = np.clip(predicted, 0, None)
    return np.sqrt(mean_squared_log_error(actual, predicted))

def evaluate_model(actual, predicted, model_name):
    predicted = np.clip(predicted, 0, None)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)
    rmsle_val = rmsle(actual, predicted)
    return {'Model': model_name, 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'RMSLE': rmsle_val}

## Model Training

We train models on the log-transformed target and convert predictions back to the original scale using expm1. This approach naturally prevents negative predictions and aligns with the RMSLE evaluation metric.

In [31]:
models = {
    'Baseline (Mean)': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.01),
    'Decision Tree': DecisionTreeRegressor(max_depth=12, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15,
                                            min_samples_leaf=3, random_state=42, n_jobs=-1),
    'Extra Trees': ExtraTreesRegressor(n_estimators=200, max_depth=15,
                                        random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                    learning_rate=0.1, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08,
                             random_state=42, n_jobs=-1),
    'AdaBoost': AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    'Bagging (DT)': BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=12),
                                      n_estimators=100, random_state=42, n_jobs=-1),
    'KNN': KNeighborsRegressor(n_neighbors=10, n_jobs=-1),
}

all_results = []
model_predictions = {}

print("Model Training Results (predicting log-count, evaluated on original scale)")
print("=" * 85)

for name, model in models.items():
    # Train on log-transformed target
    model.fit(X_train_proc, y_train_log)

    # Predict and convert back
    y_pred_log = model.predict(X_val_proc)
    y_pred = np.expm1(y_pred_log)
    y_pred = np.clip(y_pred, 0, None)

    model_predictions[name] = y_pred
    res = evaluate_model(y_val.values, y_pred, name)
    all_results.append(res)

    print("{:25s}  RMSLE={:.4f}  RMSE={:.2f}  MAE={:.2f}  R2={:.4f}".format(
        name, res['RMSLE'], res['RMSE'], res['MAE'], res['R2']))

Model Training Results (predicting log-count, evaluated on original scale)
Baseline (Mean)            RMSLE=1.4348  RMSE=204.40  MAE=143.06  R2=-0.2658
Linear Regression          RMSLE=0.8681  RMSE=177.37  MAE=102.39  R2=0.0469
Ridge Regression           RMSLE=0.8681  RMSE=177.21  MAE=102.35  R2=0.0486
Lasso Regression           RMSLE=0.8711  RMSE=164.52  MAE=99.31  R2=0.1800
Decision Tree              RMSLE=0.4126  RMSE=58.04  MAE=34.84  R2=0.8979
Random Forest              RMSLE=0.3102  RMSE=43.07  MAE=26.32  R2=0.9438
Extra Trees                RMSLE=0.3011  RMSE=38.52  MAE=23.91  R2=0.9550
Gradient Boosting          RMSLE=0.2841  RMSE=39.16  MAE=23.98  R2=0.9535
XGBoost                    RMSLE=0.2811  RMSE=37.25  MAE=22.89  R2=0.9580
AdaBoost                   RMSLE=0.6323  RMSE=126.47  MAE=76.12  R2=0.5154
Bagging (DT)               RMSLE=0.3187  RMSE=47.47  MAE=28.71  R2=0.9317
KNN                        RMSLE=0.4602  RMSE=80.77  MAE=49.65  R2=0.8023


### Prediction Error Visualizations

In [33]:
fig, axes = plt.subplots(3, 4, figsize=(20, 13))
axes = axes.flatten()

for idx, (name, pred) in enumerate(model_predictions.items()):
    if idx >= 12:
        break
    axes[idx].scatter(y_val.values, pred, alpha=0.2, s=8, color='steelblue')
    max_val = max(y_val.max(), pred.max())
    axes[idx].plot([0, max_val], [0, max_val], 'r--', linewidth=1, alpha=0.7)
    axes[idx].set_title(name, fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Actual')
    axes[idx].set_ylabel('Predicted')

plt.suptitle('Actual vs Predicted -- All Models', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Stacking Ensemble

The stacking regressor trains a meta-model on the out-of-fold predictions of multiple base learners, often capturing patterns that individual models miss.

In [35]:
stacking_reg = StackingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(n_estimators=150, max_depth=15, random_state=42, n_jobs=-1)),
        ('xgb', XGBRegressor(n_estimators=250, max_depth=6, learning_rate=0.08,
                              random_state=42, n_jobs=-1)),
        ('gb', GradientBoostingRegressor(n_estimators=150, max_depth=5,
                                          learning_rate=0.1, random_state=42)),
        ('et', ExtraTreesRegressor(n_estimators=150, max_depth=15, random_state=42, n_jobs=-1)),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5, n_jobs=-1
)

stacking_reg.fit(X_train_proc, y_train_log)
y_pred_stack = np.expm1(stacking_reg.predict(X_val_proc))
y_pred_stack = np.clip(y_pred_stack, 0, None)

res = evaluate_model(y_val.values, y_pred_stack, 'Stacking Ensemble')
all_results.append(res)
print("Stacking Ensemble:  RMSLE={:.4f}  RMSE={:.2f}  MAE={:.2f}  R2={:.4f}".format(
    res['RMSLE'], res['RMSE'], res['MAE'], res['R2']))

Stacking Ensemble:  RMSLE=0.2760  RMSE=36.22  MAE=22.20  R2=0.9603


## Voting Ensemble

In [37]:
voting_reg = VotingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)),
        ('xgb', XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08,
                              random_state=42, n_jobs=-1)),
        ('gb', GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                          learning_rate=0.1, random_state=42)),
    ],
    n_jobs=-1
)

voting_reg.fit(X_train_proc, y_train_log)
y_pred_vote = np.expm1(voting_reg.predict(X_val_proc))
y_pred_vote = np.clip(y_pred_vote, 0, None)

res = evaluate_model(y_val.values, y_pred_vote, 'Voting Ensemble')
all_results.append(res)
print("Voting Ensemble:  RMSLE={:.4f}  RMSE={:.2f}  MAE={:.2f}  R2={:.4f}".format(
    res['RMSLE'], res['RMSE'], res['MAE'], res['R2']))

Voting Ensemble:  RMSLE=0.2795  RMSE=37.54  MAE=22.81  R2=0.9573


## Cross-Validation on Top Models

In [39]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15,
                                            min_samples_leaf=3, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08,
                             random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                    learning_rate=0.1, random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=200, max_depth=15,
                                        random_state=42, n_jobs=-1),
}

X_full_proc = preprocessor.fit_transform(X)

print("5-Fold Cross-Validation (neg RMSE on log-transformed target)")
print("=" * 65)
for name, model in cv_models.items():
    scores = cross_val_score(model, X_full_proc, y_log, cv=cv,
                             scoring='neg_root_mean_squared_error', n_jobs=-1)
    rmse_scores = -scores
    print("{:25s}  Mean RMSE(log): {:.4f}  Std: {:.4f}  Folds: {}".format(
        name, rmse_scores.mean(), rmse_scores.std(),
        [round(s, 4) for s in rmse_scores]))

5-Fold Cross-Validation (neg RMSE on log-transformed target)
Random Forest              Mean RMSE(log): 0.3071  Std: 0.0038  Folds: [0.31, 0.3078, 0.3116, 0.301, 0.3049]
XGBoost                    Mean RMSE(log): 0.2748  Std: 0.0043  Folds: [0.2811, 0.268, 0.2772, 0.2737, 0.2739]
Gradient Boosting          Mean RMSE(log): 0.2808  Std: 0.0041  Folds: [0.2842, 0.2729, 0.2832, 0.2817, 0.2821]
Extra Trees                Mean RMSE(log): 0.2991  Std: 0.0043  Folds: [0.302, 0.2944, 0.3061, 0.2962, 0.2969]


## Feature Importance Analysis

In [41]:
rf_model = models['Random Forest']
xgb_model = models['XGBoost']

all_feat_names = continuous_feats + categorical_feats

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Random Forest
rf_imp = pd.DataFrame({'Feature': all_feat_names,
                        'Importance': rf_model.feature_importances_})
rf_imp = rf_imp.sort_values('Importance', ascending=True)
axes[0].barh(rf_imp['Feature'], rf_imp['Importance'], color='steelblue', edgecolor='black')
axes[0].set_title('Random Forest', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importance')

# XGBoost
xgb_imp = pd.DataFrame({'Feature': all_feat_names,
                          'Importance': xgb_model.feature_importances_})
xgb_imp = xgb_imp.sort_values('Importance', ascending=True)
axes[1].barh(xgb_imp['Feature'], xgb_imp['Importance'], color='coral', edgecolor='black')
axes[1].set_title('XGBoost', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance')

plt.suptitle('Feature Importance Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Top 5 features (Random Forest):")
print(rf_imp.sort_values('Importance', ascending=False).head().to_string(index=False))

Top 5 features (Random Forest):
     Feature  Importance
        hour    0.646010
    time_bin    0.088302
        temp    0.047696
is_rush_hour    0.040404
        year    0.029477


## Comprehensive Results

In [43]:
results_df = pd.DataFrame(all_results).sort_values('RMSLE').reset_index(drop=True)

print("All Models Ranked by RMSLE (lower is better):")
print(results_df.to_string(index=False))
print()
print("Best model: {} with RMSLE = {:.4f}".format(
    results_df.iloc[0]['Model'], results_df.iloc[0]['RMSLE']))

All Models Ranked by RMSLE (lower is better):
            Model       RMSE        MAE        R2    RMSLE
Stacking Ensemble  36.215480  22.204472  0.960264 0.275975
  Voting Ensemble  37.540379  22.811772  0.957304 0.279538
          XGBoost  37.251876  22.889111  0.957957 0.281087
Gradient Boosting  39.156814  23.977110  0.953547 0.284138
      Extra Trees  38.523551  23.910529  0.955038 0.301139
    Random Forest  43.073086  26.320873  0.943791 0.310166
     Bagging (DT)  47.471513  28.707106  0.931725 0.318692
    Decision Tree  58.040555  34.835236  0.897939 0.412587
              KNN  80.771765  49.650899  0.802342 0.460228
         AdaBoost 126.471442  76.116369  0.515404 0.632329
 Ridge Regression 177.206088 102.346504  0.048624 0.868144
Linear Regression 177.370199 102.394013  0.046861 0.868146
 Lasso Regression 164.515504  99.311961  0.180010 0.871082
  Baseline (Mean) 204.401370 143.060850 -0.265792 1.434800

Best model: Stacking Ensemble with RMSLE = 0.2760


In [44]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_n = min(12, len(results_df))
top = results_df.head(top_n)

axes[0].barh(top['Model'], top['RMSLE'], color='steelblue', edgecolor='black')
axes[0].set_title('RMSLE (lower is better)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('RMSLE')
axes[0].invert_yaxis()

axes[1].barh(top['Model'], top['R2'], color='coral', edgecolor='black')
axes[1].set_title('R-squared (higher is better)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('R2 Score')
axes[1].invert_yaxis()

plt.suptitle('Final Model Performance Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Test Set Predictions

In [46]:
# Prepare test features
X_test_final = test[feature_cols].copy()
X_test_proc_final = preprocessor.transform(X_test_final)

# Use the best performing model (XGBoost retrained on full training data)
best_model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08,
                           random_state=42, n_jobs=-1)
best_model.fit(X_full_proc, y_log)

test_pred_log = best_model.predict(X_test_proc_final)
test_pred = np.expm1(test_pred_log)
test_pred = np.clip(test_pred, 0, None).round(0).astype(int)

# Create submission
submission = pd.DataFrame({
    'datetime': test['datetime'],
    'count': test_pred
})

print("Test predictions generated: {} rows".format(len(submission)))
print()
print("Prediction statistics:")
print("  Mean: {:.1f}".format(submission['count'].mean()))
print("  Std:  {:.1f}".format(submission['count'].std()))
print("  Min:  {}".format(submission['count'].min()))
print("  Max:  {}".format(submission['count'].max()))
print()
submission.head(10)

Test predictions generated: 6493 rows

Prediction statistics:
  Mean: 184.8
  Std:  174.0
  Min:  0
  Max:  953



,datetime,count
0,2011-01-20 00:00:00,10
1,2011-01-20 01:00:00,4
2,2011-01-20 02:00:00,3
3,2011-01-20 03:00:00,2
4,2011-01-20 04:00:00,1
5,2011-01-20 05:00:00,6
6,2011-01-20 06:00:00,31
7,2011-01-20 07:00:00,90
8,2011-01-20 08:00:00,207
9,2011-01-20 09:00:00,110


In [47]:
# Visualize test predictions distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(submission['count'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Test Predictions', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Count')
axes[0].set_ylabel('Frequency')

# Compare train actual vs test predicted
axes[1].hist(train['count'], bins=50, alpha=0.5, color='steelblue', edgecolor='black',
             label='Train (Actual)', density=True)
axes[1].hist(submission['count'], bins=50, alpha=0.5, color='coral', edgecolor='black',
             label='Test (Predicted)', density=True)
axes[1].set_title('Train Actual vs Test Predicted Distributions', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

## Conclusions and Insights

**Demand Drivers**

Hour of day is by far the most important predictor, reflecting the strong commute-driven demand pattern on working days (peaks at 8 AM and 5-6 PM) and leisure-driven demand on weekends (midday peak). Temperature shows a positive correlation with rentals, while humidity and adverse weather conditions suppress demand.

**Modeling Approach**

Log-transforming the target variable was essential due to the heavy right skew. Tree-based ensemble models (XGBoost, Random Forest, Gradient Boosting) consistently outperformed linear models, capturing the non-linear interactions between temporal and weather features. The stacking and voting ensembles provided marginal additional gains by combining diverse model perspectives.

**Temporal Feature Engineering**

Extracting hour, day of week, month, year, time bins, and rush hour flags from the raw datetime significantly improved model performance. These features encode the cyclical patterns that drive bike sharing demand far better than the raw timestamp.

**Recommendations for Deployment**

- The XGBoost or stacking ensemble model can be deployed for hourly demand forecasting to support station rebalancing operations.
- Integrate real-time weather forecast data for forward-looking predictions.
- Consider adding event calendar data (sports events, concerts, protests) as additional features.
- Retrain the model periodically as new data accumulates and usage patterns evolve with system expansion.

---

End of Notebook